# Havnsø – CO₂ Storage Capacity Assessment

This notebook runs the main probabilistic static assessment for **Havnsø Scenario 1 – Gassum Formation**:

$$SC = GRV \times (N/G) \times \phi \times \rho_{CO_2} \times S_{eff}$$

Change the values in the **Editable inputs** cell, then choose **Runtime → Run all**. The layout and workflow match the Rødby notebook.

In [ ]:
#@title Install dependencies { display-mode: "form" }
# Install the latest package and plotting tools from GitHub.
%pip install -q "git+https://github.com/AnaSoles/ggg-co2-storage-eval.git" matplotlib pandas

In [ ]:
#@title Import libraries { display-mode: "form" }
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from storageeval import Distribution, StorageSite, simulate

plt.style.use("seaborn-v0_8-whitegrid")

## Editable inputs

Enter minimum, most likely, and maximum values. Fractions such as porosity must be decimals: `0.219` means 21.9%. The values below come from GEUS Report 2023/38, Scenario 1 input Table 8.2.3 (report page 152).

**Evidence status:** Havnsø itself was not penetrated by a well in this study. GRV comes from updated seismic mapping; net-to-gross and porosity are prognoses based mainly on Stenlille and surrounding wells; CO₂ density is calculated from assumed hydrostatic pressure and temperature; storage efficiency is an uncalibrated screening assumption.

In [ ]:
site_name = "Havnsø – Gassum Formation – Scenario 1"
iterations = 100_000
random_seed = 42

#                       minimum, most likely, maximum
grv_km3 =               (2.9,    5.0,    8.0)
net_to_gross =          (0.60,   0.75,   0.90)
porosity =              (0.175,  0.219,  0.263)
co2_density_kg_m3 =     (663.86, 698.8,  768.68)
storage_efficiency =    (0.05,   0.10,   0.20)

## Input parameter table

This table is generated from the editable values above, so it updates automatically when you change an input. `Description / interpretation`, `Input type`, and `Source / basis` keep measured, interpreted, calculated and assumed values visually separate.

In [ ]:
input_table = pd.DataFrame([
    ["Gross rock volume (GRV)", "km³", "PERT", *grv_km3, "Seismic-derived estimate", "Volume of the Gassum Formation reservoir within the mapped Havnsø four-way closure. The range captures mapping, seismic tie and depth-conversion uncertainty.", "GEUS Report 2023/38, Sections 8.1–8.2.1 and Tables 8.2.1/8.2.3"],
    ["Net-to-gross (N/G)", "fraction", "PERT", *net_to_gross, "Analogue / well-derived prognosis", "Fraction of the gross interval interpreted as reservoir sandstone. Scenario 1 uses Stenlille-19 and surrounding wells as analogues because no well penetrates the Havnsø reservoir.", "GEUS Report 2023/38, Sections 7.1 and 8.2.2; Tables 7.1.5 and 8.2.3"],
    ["Porosity (φ)", "fraction", "PERT", *porosity, "Analogue petrophysical prognosis", "Average effective porosity predicted from Stenlille and surrounding-well petrophysics, with approximately ±20% uncertainty.", "GEUS Report 2023/38, Sections 7.1 and 8.2.3; Tables 7.1.5 and 8.2.3"],
    ["In-situ CO₂ density", "kg/m³", "PERT", *co2_density_kg_m3, "Thermodynamic estimate", "Calculated at approximately 15.99 MPa and 52°C. The PERT range applies −5%/+10% around the 698.8 kg/m³ mode; it is not a direct measurement.", "GEUS Report 2023/38, Section 8.2.4 and Tables 8.2.2/8.2.3; Span & Wagner (1996)"],
    ["Storage efficiency", "fraction", "PERT", *storage_efficiency, "Literature-informed screening assumption", "Assumed fraction of pore volume effectively occupied by CO₂. The 10% mode is not a Havnsø measurement and should be refined with site-specific dynamic simulation.", "GEUS Report 2023/38, Section 8.2.5 and Table 8.2.3; Wang et al. (2013)"],
], columns=["Parameter", "Unit", "Distribution", "Minimum", "Mode", "Maximum", "Input type", "Description / interpretation", "Source / basis"])
input_table

## Havnsø input-data provenance

The five independent PERT inputs follow Gregersen et al. (2023), GEUS Report 2023/38, Scenario 1:

$$SC = GRV \times (N/G) \times \phi \times \rho_{CO_2} \times S_{eff}$$

Scenario 1 is the report's highest-capacity updated static case. It should not be confused with the older static assessment in GEUS 2020/46 or the preliminary version-0 dynamic model in GEUS 2020/48.

### References

- Gregersen, U., Vosgerau, H., Smit, F.W.H., et al. (2023). *CCS2022-2024 WP1: The Havnsø structure – Seismic data and interpretation to mature potential geological storage of CO₂*. GEUS Report 2023/38. [https://doi.org/10.22008/gpub/34705](https://doi.org/10.22008/gpub/34705)
- Hjelm, L., Anthonsen, K.L., Dideriksen, K., et al. (2020). *Capture, Storage and Use of CO₂ (CCUS): Evaluation of the CO₂ storage potential in Denmark*. GEUS Report 2020/46.
- Nielsen, C.M., Frykman, P., Dalhoff, F., et al. (2020). *Dynamic storage capacity assessment of the Havnsø and Hanstholm structures*. GEUS Report 2020/48.
- Span, R. & Wagner, W. (1996). *A New Equation of State for Carbon Dioxide*. *Journal of Physical and Chemical Reference Data, 25*(6), 1509–1596. [https://doi.org/10.1063/1.555991](https://doi.org/10.1063/1.555991)
- Wang, Y., Zhang, K. & Wu, N. (2013). *Numerical Investigation of the Storage Efficiency Factor for CO₂ Geological Sequestration in Saline Formations*. *Energy Procedia, 37*, 5267–5274. [https://doi.org/10.1016/j.egypro.2013.06.443](https://doi.org/10.1016/j.egypro.2013.06.443)

In [ ]:
site = StorageSite(
    name=site_name,
    grv=Distribution.pert(*grv_km3),
    net_to_gross=Distribution.pert(*net_to_gross),
    porosity=Distribution.pert(*porosity),
    co2_density=Distribution.pert(*co2_density_kg_m3),
    storage_efficiency=Distribution.pert(*storage_efficiency),
)

result = simulate(site, iterations=iterations, seed=random_seed)
summary = result.summary()
report_values = {"p90_mt": 41.25, "p50_mt": 62.82, "p10_mt": 90.42, "mean_mt": 64.81}
result_rows = [("P90 (conservative)", "p90_mt"), ("P50 (median)", "p50_mt"), ("P10 (upside)", "p10_mt"), ("Mean", "mean_mt")]
comparison = pd.DataFrame({
    "Notebook (Mt CO₂)": [summary[key] for _, key in result_rows],
    "GEUS Table 8.3.1 (Mt CO₂)": [report_values[key] for _, key in result_rows],
}, index=[label for label, _ in result_rows])
comparison["Difference (Mt CO₂)"] = comparison["Notebook (Mt CO₂)"] - comparison["GEUS Table 8.3.1 (Mt CO₂)"]
comparison.round(2)

The mean should be reproduced closely. Differences of a few Mt in P90/P50/P10 are expected because GEUS Report 2023/38 does not publish its Monte Carlo iteration count, random seed, or exact PERT implementation. The comparison is therefore a transparent validation check, not a forced calibration.

## Published Havnsø scenario comparison

The report defines three geological reservoir scenarios. Scenario 1 is the main study because it has the highest predicted net sandstone thickness and the clearest valid PERT input table.

In [ ]:
published_scenarios = pd.DataFrame([
    ["Scenario 1 – main", 111.0, 0.75, 0.219, 805, 41.25, 62.82, 90.42, 64.81],
    ["Scenario 2", 69.0, 0.63, 0.233, 1087, 37.04, 56.34, 82.80, 58.43],
    ["Scenario 3", 47.6, 0.33, 0.273, 1516, np.nan, np.nan, np.nan, 35.0],
], columns=["Scenario", "Net sandstone (m)", "N/G mode", "Porosity mode", "Permeability (mD)", "P90 (Mt)", "P50 (Mt)", "P10 (Mt)", "Mean (Mt)"])
published_scenarios

# Important: Table 8.2.5 prints Scenario 3 N/G as 0.60 / 0.33 / 0.90,
# which is not a valid minimum/mode/maximum order. It is not sampled here.

## Input uncertainty distributions

In [ ]:
#@title Show input uncertainty distributions { display-mode: "form" }
labels = {
    "grv_km3": "GRV (km³)",
    "net_to_gross": "Net-to-gross",
    "porosity": "Porosity",
    "co2_density_kg_m3": "CO₂ density (kg/m³)",
    "storage_efficiency": "Storage efficiency",
}
input_ranges = {
    "grv_km3": grv_km3,
    "net_to_gross": net_to_gross,
    "porosity": porosity,
    "co2_density_kg_m3": co2_density_kg_m3,
    "storage_efficiency": storage_efficiency,
}
line_styles = [
    ("Minimum", "#1f77b4", ":"),
    ("Mode", "#2ca02c", "--"),
    ("Mean", "#ff7f0e", "-"),
    ("Maximum", "#d62728", ":"),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (name, values) in zip(axes.flat, result.inputs.items()):
    minimum, mode, maximum = input_ranges[name]
    mean = float(np.mean(values))
    reference_values = [minimum, mode, mean, maximum]
    ax.hist(values, bins=45, color="#b9d7f0", edgecolor="white", alpha=0.9)
    for (line_name, color, linestyle), reference in zip(line_styles, reference_values):
        ax.axvline(
            reference,
            color=color,
            linestyle=linestyle,
            linewidth=1.6,
            label=f"{line_name}: {reference:.4g}",
        )
    ax.set_title(labels[name])
    ax.set_ylabel("Simulations")
    ax.legend(fontsize=8, frameon=True, loc="upper right")
axes.flat[-1].axis("off")
fig.suptitle(f"{site_name} – input uncertainty distributions and values used", fontsize=15)
fig.tight_layout()
plt.show()


## Storage-capacity probability distribution

Grey bars show the simulated Havnsø Scenario 1 capacities, the orange line is a fitted lognormal probability density, and the red curve is cumulative exceedance probability. The P90, P50 and P10 markers use the same Monte Carlo result reported in the comparison table.


In [ ]:
#@title Show capacity histogram with fitted density and cumulative curve { display-mode: "form" }
def plot_capacity_distribution(ax_density, result_name, result):
    values = np.asarray(result.capacity_mt, dtype=float)
    summary = result.summary()

    ax_density.hist(
        values,
        bins=50,
        density=True,
        color="#d9d9d9",
        edgecolor="white",
        linewidth=0.5,
        label="Simulated capacity",
    )

    log_values = np.log(values)
    mu = float(np.mean(log_values))
    sigma = float(np.std(log_values, ddof=1))
    x = np.linspace(float(np.min(values)), float(np.max(values)), 500)
    fitted_density = np.exp(
        -0.5 * ((np.log(x) - mu) / sigma) ** 2
    ) / (x * sigma * np.sqrt(2 * np.pi))
    ax_density.plot(x, fitted_density, color="#f39c12", linewidth=2.0, label="Fitted density")

    ax_cumulative = ax_density.twinx()
    capacity_sorted = np.sort(values)
    exceedance_pct = (
        1 - np.arange(1, capacity_sorted.size + 1) / (capacity_sorted.size + 1)
    ) * 100
    ax_cumulative.plot(
        capacity_sorted,
        exceedance_pct,
        color="#e31a1c",
        linewidth=1.8,
        label="Cumulative exceedance",
    )

    markers = [
        ("P90", "p90_mt", 90, "#b2182b"),
        ("P50", "p50_mt", 50, "#ff8c42"),
        ("P10", "p10_mt", 10, "#b2182b"),
    ]
    for marker_label, key, probability, color in markers:
        marker_value = summary[key]
        ax_density.axvline(
            marker_value,
            color=color,
            linestyle="--",
            linewidth=1.1,
            alpha=0.9,
        )
        ax_cumulative.plot(marker_value, probability, "o", color=color, markersize=4)
        vertical_offset = 8 if marker_label != "P10" else -14
        ax_cumulative.annotate(
            f"{marker_label}: {marker_value:.1f} Mt",
            (marker_value, probability),
            xytext=(5, vertical_offset),
            textcoords="offset points",
            color=color,
            fontsize=9,
        )

    ax_density.text(
        0.98,
        0.96,
        f"Mean: {summary['mean_mt']:.1f} Mt\nSD: {np.std(values, ddof=1):.1f} Mt",
        transform=ax_density.transAxes,
        ha="right",
        va="top",
        color="#b56500",
        fontsize=9,
    )
    ax_density.set_title(result_name)
    ax_density.set_xlabel("Storage capacity (Mt CO₂)")
    ax_density.set_ylabel("Probability density")
    ax_cumulative.set_ylabel("Exceedance probability (%)", color="#e31a1c")
    ax_cumulative.set_ylim(0, 100)
    ax_cumulative.tick_params(axis="y", colors="#e31a1c")

    density_handles, density_labels = ax_density.get_legend_handles_labels()
    cumulative_handles, cumulative_labels = ax_cumulative.get_legend_handles_labels()
    ax_density.legend(
        density_handles + cumulative_handles,
        density_labels + cumulative_labels,
        loc="upper center",
        fontsize=8,
    )

fig, ax = plt.subplots(figsize=(11, 7))
plot_capacity_distribution(ax, site_name, result)
fig.tight_layout()
plt.show()


## Exceedance curve

P90 is the capacity that has a 90% probability of being exceeded; P10 is the upside estimate.

In [ ]:
#@title Show exceedance curve { display-mode: "form" }
fig, ax = result.plot_exceedance()
plt.show()

## Linear capacity confidence ranges

The colored bar summarizes conservative, central, and upside capacity ranges. These are probabilistic static capacity estimates, not booked reserves.

In [ ]:
#@title Show linear capacity confidence ranges { display-mode: "form" }
fig, ax = result.plot_capacity_ranges()
plt.show()

## One-at-a-time capacity tornado

For each bar, one input moves from its P90 input value (10th sample percentile) to its P10 input value (90th sample percentile), while the other four inputs remain at their sampled means. The labels show the resulting low and high storage capacities. This measures direct capacity impact and complements—not replaces—the Spearman rank chart below.


In [ ]:
#@title Show one-at-a-time capacity tornado { display-mode: "form" }
def capacity_from_inputs(parameter_values):
    return float(np.prod(list(parameter_values.values())))

sample_means = {
    name: float(np.mean(samples))
    for name, samples in result.inputs.items()
}
baseline_capacity = capacity_from_inputs(sample_means)
tornado_rows = []

for name, samples in result.inputs.items():
    low_inputs = sample_means.copy()
    high_inputs = sample_means.copy()
    low_inputs[name] = float(np.quantile(samples, 0.10))
    high_inputs[name] = float(np.quantile(samples, 0.90))
    tornado_rows.append(
        (
            labels[name],
            capacity_from_inputs(low_inputs),
            capacity_from_inputs(high_inputs),
        )
    )

ordered_rows = sorted(tornado_rows, key=lambda row: row[2] - row[1])
parameter_names = [row[0] for row in ordered_rows]
low_capacities = np.asarray([row[1] for row in ordered_rows])
high_capacities = np.asarray([row[2] for row in ordered_rows])
bar_widths = high_capacities - low_capacities

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(
    parameter_names,
    bar_widths,
    left=low_capacities,
    color="#5b9bd5",
    edgecolor="#2f5597",
    alpha=0.9,
)
ax.axvline(
    baseline_capacity,
    color="#c00000",
    linestyle="--",
    linewidth=1.4,
    label=f"Base: {baseline_capacity:.1f} Mt",
)

chart_range = max(
    float(np.max(high_capacities) - np.min(low_capacities)),
    1e-12,
)
padding = 0.10 * chart_range
ax.set_xlim(
    float(np.min(low_capacities) - padding),
    float(np.max(high_capacities) + padding),
)

for bar, low, high, width in zip(
    bars,
    low_capacities,
    high_capacities,
    bar_widths,
):
    y = bar.get_y() + bar.get_height() / 2
    if width >= 0.16 * chart_range:
        ax.text(
            low + 0.03 * width,
            y,
            f"{low:.1f}",
            ha="left",
            va="center",
            color="white",
            fontsize=9,
            fontweight="bold",
        )
        ax.text(
            high - 0.03 * width,
            y,
            f"{high:.1f}",
            ha="right",
            va="center",
            color="white",
            fontsize=9,
            fontweight="bold",
        )
    else:
        ax.annotate(
            f"{low:.1f}",
            (low, y),
            xytext=(-4, 0),
            textcoords="offset points",
            ha="right",
            va="center",
            color="#1f1f1f",
            fontsize=8,
            fontweight="bold",
        )
        ax.annotate(
            f"{high:.1f}",
            (high, y),
            xytext=(4, 0),
            textcoords="offset points",
            ha="left",
            va="center",
            color="#1f1f1f",
            fontsize=8,
            fontweight="bold",
        )

ax.set_title(f"{site_name} – one-at-a-time capacity sensitivity")
ax.set_xlabel("Storage capacity (Mt CO₂)")
ax.legend(fontsize=9, loc="lower right")
fig.tight_layout()
plt.show()


## Spearman rank sensitivity

This second chart retains the original global sensitivity view. Longer bars identify inputs with the strongest monotonic association with simulated capacity; the printed value is the Spearman rank-correlation coefficient.


In [ ]:
#@title Show Spearman rank sensitivity { display-mode: "form" }
fig, ax = result.plot_sensitivity()
fig.set_size_inches(10, 6)

readable_tick_labels = [labels.get(tick.get_text(), tick.get_text()) for tick in ax.get_yticklabels()]
ax.set_yticks(ax.get_yticks(), labels=readable_tick_labels)

for bar in ax.patches:
    value = float(bar.get_width())
    y = bar.get_y() + bar.get_height() / 2
    if abs(value) >= 0.14:
        inset = 0.025 if value >= 0 else -0.025
        ax.text(
            value - inset,
            y,
            f"{value:.2f}",
            ha="right" if value >= 0 else "left",
            va="center",
            color="white",
            fontsize=9,
            fontweight="bold",
        )
    else:
        offset = 5 if value >= 0 else -5
        ax.annotate(
            f"{value:.2f}",
            (value, y),
            xytext=(offset, 0),
            textcoords="offset points",
            ha="left" if value >= 0 else "right",
            va="center",
            color="#1f1f1f",
            fontsize=9,
            fontweight="bold",
        )

plt.show()


## Evidence reserved for the technical-risk stage

This notebook intentionally calculates **static capacity only**. GEUS Report 2020/48 provides separate preliminary dynamic evidence for Havnsø: a version-0 Eclipse 100 model, three wells injecting 1 Mt/year each for about 90 years, and approximately 270 Mt injected before the model pressure constraint. Those values come from an older analogue-based model and are not combined numerically with the updated 2023 static inputs here.

The next risk module should treat capacity, injectivity/pressure and containment as separate success criteria, with clear provenance for every borrowed or assumed dynamic input.

## Important limitation

This is a static volumetric screening assessment. It does not yet represent pressure constraints, injectivity, plume migration, dynamic reservoir simulation, or economics.